# 03. Torch FX Graph Capture and Custom Compiler Passes

- **Category:** 12_compilation_and_performance
- **Lineage & Citations:** Reed et al. (2021 *Torch.fx: Practical program capture and transformation for deep learning in Python*); Ioffe & Szegedy (2015 *Batch Normalization: Accelerating Deep Network Training*); Vaswani et al. (2017 *Attention Is All You Need*).
- **Purpose:** Formulate and verify Torch.FX program capture, DAG Intermediate Representation (IR), and custom graph compiler passes: algebraic Linear-BatchNorm folding and horizontal GEMM concatenation.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [02. Dynamic Shapes Mark Dynamic and Recompilation Guards](02_dynamic_shapes_mark_dynamic_and_recompilation_guards.ipynb) $\to$ `🔗 Downstream Syntheses:` [04. Torch Export AOT Autograd and IR Inspection](04_torch_export_aot_autograd_and_ir_inspection.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not run symbolic FX tracing on modules containing data-dependent control flow without TorchDynamo fallback; do not fold BatchNorm during training mode (only apply folding when weights are frozen for inference).

## Mathematical Foundations & Compiler Transformations

1. **Linear + BatchNorm Algebraic Folding**:
   $$\mathbf{W}_{\text{fold}} = \mathbf{W} \odot \left( \frac{\boldsymbol{\gamma}}{\sqrt{\boldsymbol{\sigma}^2 + \epsilon}} \right)^T, \quad \mathbf{b}_{\text{fold}} = (\mathbf{b} - \boldsymbol{\mu}) \odot \frac{\boldsymbol{\gamma}}{\sqrt{\boldsymbol{\sigma}^2 + \epsilon}} + \boldsymbol{\beta}$$
2. **Folded Forward Equivalence**:
   $$\mathbf{y} = \mathbf{x} \mathbf{W}_{\text{fold}} + \mathbf{b}_{\text{fold}} \equiv \text{BN}(\mathbf{x} \mathbf{W} + \mathbf{b})$$
3. **Horizontal Multi-Head GEMM Fusion**:
   $$\mathbf{W}_{\text{fused}} = [\mathbf{W}_1 \;\|\; \mathbf{W}_2 \;\|\; \mathbf{W}_3], \quad \mathbf{b}_{\text{fused}} = [\mathbf{b}_1 \;\|\; \mathbf{b}_2 \;\|\; \mathbf{b}_3]$$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/12_compilation_and_performance/03_torch_fx_graph_capture_and_custom_compiler_passes.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def algebraic_linear_bn_folding(w, b, mean, var, gamma, beta, eps=1e-5):
    scale = gamma / torch.sqrt(var + float(eps))
    w_fold = w * scale.unsqueeze(0)
    b_fold = (b - mean) * scale + beta
    return w_fold, b_fold

def sequential_linear_bn(x, w, b, mean, var, gamma, beta, eps=1e-5):
    z = x @ w + b
    return ((z - mean) / torch.sqrt(var + float(eps))) * gamma + beta

def parallel_gemm_fusion(w1, b1, w2, b2, w3, b3):
    return torch.cat([w1, w2, w3], dim=-1), torch.cat([b1, b2, b3], dim=-1)

In [ ]:
# Verify algebraic folding against golden reference
w_fold, b_fold = algebraic_linear_bn_folding(
    weights["lin_w"], weights["lin_b"], weights["bn_mean"], weights["bn_var"],
    weights["bn_gamma"], weights["bn_beta"], eps=1e-5
)
assert torch.max(torch.abs(w_fold - weights["w_fold_ref"])) < 1e-5
assert torch.max(torch.abs(b_fold - weights["b_fold_ref"])) < 1e-5

folded_out = weights["lin_x"] @ w_fold + b_fold
assert torch.max(torch.abs(folded_out - weights["folded_out_ref"])) < 1e-5

# Verify horizontal GEMM fusion
w_fused, b_fused = parallel_gemm_fusion(
    weights["w1"], weights["b1"], weights["w2"], weights["b2"], weights["w3"], weights["b3"]
)
assert torch.max(torch.abs(w_fused - weights["w_fused_ref"])) < 1e-5
assert torch.max(torch.abs(b_fused - weights["b_fused_ref"])) < 1e-5
print("Linear-BN folding and Parallel GEMM fusion parity verified!")

In [ ]:
# Benchmark sequential Linear-BN vs folded Linear forward pass
x = weights["lin_x"]
t0 = time.perf_counter()
for _ in range(500):
    _ = sequential_linear_bn(
        x, weights["lin_w"], weights["lin_b"], weights["bn_mean"],
        weights["bn_var"], weights["bn_gamma"], weights["bn_beta"], 1e-5
    )
t_seq = (time.perf_counter() - t0) / 500 * 1e6

t1 = time.perf_counter()
for _ in range(500):
    _ = x @ w_fold + b_fold
t_fold = (time.perf_counter() - t1) / 500 * 1e6
print(f"Sequential Linear+BN: {t_seq:.2f} us | Folded Linear: {t_fold:.2f} us (Speedup: {t_seq/t_fold:.2f}x)")

In [ ]:
# Equivalence check: Folded Linear vs Sequential Execution across random inputs
test_x = torch.randn(16, 4)
out_unfolded = sequential_linear_bn(
    test_x, weights["lin_w"], weights["lin_b"], weights["bn_mean"],
    weights["bn_var"], weights["bn_gamma"], weights["bn_beta"], 1e-5
)
out_folded = test_x @ w_fold + b_fold
max_discrepancy = float(torch.max(torch.abs(out_unfolded - out_folded)))
print(f"Transitive folded vs sequential max error: {max_discrepancy:.2e}")
assert max_discrepancy < 1e-5, f"Folding error exceeded tolerance: {max_discrepancy}"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Torch.FX Graph Capture** provides a lightweight, inspectable, and transformable Python AST intermediate representation without C++ overhead.
2. **Algebraic BN Folding** collapses inference normalization layers into static weight affine forms, eliminating memory round-trips.
3. **Horizontal Fusion** consolidates independent parallel GEMMs into a single concatenated kernel launch.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [03_torch_fx_graph_capture_and_custom_compiler_passes.md](../../markdown/12_compilation_and_performance/03_torch_fx_graph_capture_and_custom_compiler_passes.md)
- **PyTorch 2.x Module:** [03_torch_fx_graph_capture_and_custom_compiler_passes.py](../../pytorch2/12_compilation_and_performance/03_torch_fx_graph_capture_and_custom_compiler_passes.py)
- **Functional JAX Module:** [03_torch_fx_graph_capture_and_custom_compiler_passes.py](../../jax/12_compilation_and_performance/03_torch_fx_graph_capture_and_custom_compiler_passes.py)
- **Native Rust Candle Track:** [03_torch_fx_graph_capture_and_custom_compiler_passes.rs](../../candle/12_compilation_and_performance/03_torch_fx_graph_capture_and_custom_compiler_passes.rs)
- **Hardware Kernel (CPU / WGSL):** [03_torch_fx_graph_capture_and_custom_compiler_passes.rs](../../kernels/12_compilation_and_performance/03_torch_fx_graph_capture_and_custom_compiler_passes.rs)
- **Golden SafeTensors:** [03_torch_fx_graph_capture_and_custom_compiler_passes.safetensors](../../weights/12_compilation_and_performance/03_torch_fx_graph_capture_and_custom_compiler_passes.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 02. Dynamic Shapes Mark Dynamic and Recompilation Guards](02_dynamic_shapes_mark_dynamic_and_recompilation_guards.ipynb) | [Master Index](../../README.md#12_compilation_and_performance) | [Next: 04. Torch Export AOT Autograd and IR Inspection](04_torch_export_aot_autograd_and_ir_inspection.ipynb) &rarr;